[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/09_data_aggregation/09_9_Exercises.ipynb)

# 09.9: Exercises

These exercises cover the full range of tools from module 09: multi-key groupby, `agg()` with named aggregations, `transform()`, `filter()`, and `pivot_table()`. All exercises use the Gapminder dataset.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"
df = pd.read_csv(url, sep="\t")
df2007 = df[df["year"] == 2007].copy().reset_index(drop=True)

print(f"Full dataset: {df.shape}")
print(f"2007 snapshot: {df2007.shape}")
df.head()

---
## Part 1: Orientation and basic groupby

### Exercise 1

Confirm that each country appears exactly 12 times in the full dataset. Then show how many distinct countries are in each continent, sorted from most to fewest.

In [ ]:
# your code here

In [ ]:
#@title Solution
counts = df["country"].value_counts()
print("All countries appear exactly 12 times:", (counts == 12).all())
print()
df.groupby("continent")["country"].nunique().sort_values(ascending=False)

### Exercise 2

Using `df2007`, compute the mean, median, and standard deviation of `lifeExp` for each continent. Return a single DataFrame sorted by mean life expectancy (descending). Round all values to one decimal place.

In [ ]:
# your code here

In [ ]:
#@title Solution
(
    df2007.groupby("continent")["lifeExp"]
    .agg(["mean", "median", "std"])
    .round(1)
    .sort_values("mean", ascending=False)
)

### Exercise 3

Using named aggregations on `df2007`, produce a single summary table with one row per continent containing:
- `avg_life`: mean life expectancy
- `total_pop`: total population
- `avg_gdp`: mean GDP per capita
- `n_countries`: number of distinct countries

Round numeric columns to 1 decimal place and sort by `avg_life` descending.

In [ ]:
# your code here

In [ ]:
#@title Solution
(
    df2007.groupby("continent").agg(
        avg_life    = ("lifeExp",   "mean"),
        total_pop   = ("pop",       "sum"),
        avg_gdp     = ("gdpPercap", "mean"),
        n_countries = ("country",   "nunique")
    )
    .round(1)
    .sort_values("avg_life", ascending=False)
)

---
## Part 2: Multiple groupby keys

### Exercise 4

Using the full dataset, group by `["continent", "year"]` and compute the mean GDP per capita. Then call `unstack("year")` and display the result rounded to 0 decimal places. Which continent shows the largest absolute increase in mean GDP from 1952 to 2007?

In [ ]:
# your code here

In [ ]:
#@title Solution
wide = (
    df.groupby(["continent", "year"])["gdpPercap"]
    .mean()
    .unstack("year")
    .round(0)
)
print(wide)
print()
gain = wide[2007] - wide[1952]
print("Largest absolute GDP gain:")
print(gain.sort_values(ascending=False))
# Oceania (about $19,500) narrowly beats Europe (about $19,400).

### Exercise 5

Find the 5 countries with the highest life expectancy in 2007. Show their country name, continent, and life expectancy.

In [ ]:
# your code here

In [ ]:
#@title Solution
df2007.nlargest(5, "lifeExp")[["country", "continent", "lifeExp"]]

### Exercise 6

Find the 5 countries with the lowest GDP per capita in 2007. Show their country name, continent, and GDP per capita rounded to 2 decimal places.

In [ ]:
# your code here

In [ ]:
#@title Solution
result = df2007.nsmallest(5, "gdpPercap")[["country", "continent", "gdpPercap"]].copy()
result["gdpPercap"] = result["gdpPercap"].round(2)
result

---
## Part 3: `transform()`

### Exercise 7

Using the full dataset (all years), add a column `continent_avg_gdp` that holds the continent-level mean GDP per capita for each row, computed across all years. Then show the first 10 rows of `country`, `continent`, `year`, `gdpPercap`, and `continent_avg_gdp`.

In [ ]:
# your code here

In [ ]:
#@title Solution
df_ex7 = df.copy()
df_ex7["continent_avg_gdp"] = df_ex7.groupby("continent")["gdpPercap"].transform("mean")
df_ex7[["country", "continent", "year", "gdpPercap", "continent_avg_gdp"]].head(10).round(1)

### Exercise 8

Using `df2007`, compute each country's deviation from its continent average GDP per capita (`gdpPercap - continent_mean_gdp`). Then find the country in each continent that was furthest above its continental average in 2007. Return one row per continent.

In [ ]:
# your code here

In [ ]:
#@title Solution
df_ex8 = df2007.copy()
df_ex8["continent_avg_gdp"] = df_ex8.groupby("continent")["gdpPercap"].transform("mean")
df_ex8["gdp_vs_continent"] = df_ex8["gdpPercap"] - df_ex8["continent_avg_gdp"]

(
    df_ex8.loc[df_ex8.groupby("continent")["gdp_vs_continent"].idxmax()]
    [["continent", "country", "gdpPercap", "continent_avg_gdp", "gdp_vs_continent"]]
    .round(1)
    .sort_values("continent")
)

### Exercise 9

Compute a within-continent z-score for `lifeExp` in `df2007`: subtract each country's continent mean and divide by the continent standard deviation. Store the result in a column `life_zscore`. Which country had the highest z-score (most above its continental peers) in 2007?

In [ ]:
# your code here

In [ ]:
#@title Solution
df_ex9 = df2007.copy()
df_ex9["life_zscore"] = (
    (df_ex9["lifeExp"] - df_ex9.groupby("continent")["lifeExp"].transform("mean"))
    / df_ex9.groupby("continent")["lifeExp"].transform("std")
)
df_ex9.nlargest(5, "life_zscore")[["country", "continent", "lifeExp", "life_zscore"]].round(2)

---
## Part 4: `filter()`

### Exercise 10

Using the full dataset and `groupby("country").filter()`, keep only countries where life expectancy exceeded 50 in every year from 1977 onward. How many rows remain? How many distinct countries?

In [ ]:
# your code here

In [ ]:
#@title Solution
df_recent = df[df["year"] >= 1977].copy()
consistent = df_recent.groupby("country").filter(
    lambda g: g["lifeExp"].min() > 50
)
print(f"Rows remaining: {len(consistent)}")
print(f"Distinct countries: {consistent['country'].nunique()}")

### Exercise 11

Using `df2007` and `groupby("continent").filter()`, keep only continents where the mean GDP per capita exceeded $10,000. Then compute the average population of those continents across all years (using the full dataset filtered to those same continents).

In [ ]:
# your code here

In [ ]:
#@title Solution
rich_continents = (
    df2007.groupby("continent")
    .filter(lambda g: g["gdpPercap"].mean() > 10000)
    ["continent"].unique()
)
print("Qualifying continents:", rich_continents.tolist())

(
    df[df["continent"].isin(rich_continents)]
    .groupby("continent")["pop"]
    .mean()
    .round(0)
)

---
## Part 5: `pivot_table()` and visualization

### Exercise 12

Build a continent × year pivot table of mean life expectancy using `pd.pivot_table()`. Add `margins=True` so overall means appear in the `All` row and column. Round to 1 decimal place.

In [ ]:
# your code here

In [ ]:
#@title Solution
pd.pivot_table(
    df,
    values="lifeExp",
    index="continent",
    columns="year",
    aggfunc="mean",
    margins=True,
    margins_name="All"
).round(1)

### Exercise 13

Build the same continent × year life expectancy pivot table (without margins). Pass it to `sns.heatmap()` with annotations and a sequential colormap of your choice. Add a title.

In [ ]:
# your code here

In [ ]:
#@title Solution
pt = pd.pivot_table(
    df,
    values="lifeExp",
    index="continent",
    columns="year",
    aggfunc="mean"
).round(1)

sns.heatmap(pt, annot=True, fmt=".1f", cmap="YlOrRd", vmin=35, vmax=80).set(
    title="Mean life expectancy by continent and year",
    xlabel="Year",
    ylabel="Continent"
)
plt.show()


### Exercise 14 (Challenge)

For each continent, find the country that experienced the largest **increase** in life expectancy between 1952 and 2007. Return a DataFrame with one row per continent showing the continent name, the country name, and the gain in years (rounded to 1 decimal place), sorted by gain descending.

In [ ]:
# your code here

In [ ]:
#@title Solution
life_1952 = df[df["year"] == 1952][["country", "continent", "lifeExp"]].rename(columns={"lifeExp": "life_1952"})
life_2007 = df[df["year"] == 2007][["country", "lifeExp"]].rename(columns={"lifeExp": "life_2007"})

# merge() is previewed here; module 10 teaches it properly alongside SQL JOIN
gains = life_1952.merge(life_2007, on="country")
gains["gain"] = (gains["life_2007"] - gains["life_1952"]).round(1)

best_per_continent = (
    gains.loc[gains.groupby("continent")["gain"].idxmax()]
    [["continent", "country", "gain"]]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
best_per_continent

## Module 09 complete

You have now practiced the full groupby toolkit: multi-key groupby with the MultiIndex, `agg()` with named aggregations for building summary tables, `transform()` for adding group-level statistics back to every row, `filter()` for keeping or removing entire groups by condition, and `pivot_table()` for producing wide-format summaries directly. These tools answer the questions that a single-key groupby cannot: how a group changes over time, how each member of a group compares to its peers, and which groups meet a threshold worth analyzing further.

Module 10 introduces a second language for the same ideas. SQL has been the standard tool for querying tabular data in databases for decades, and pandas borrows its vocabulary directly: `WHERE` maps to boolean indexing, `GROUP BY` maps to `groupby()`, and `JOIN` maps to `merge()`. Working through both syntaxes side by side sharpens each one, because you can see exactly where they agree and where they differ.